# SPILLTRACE — U-Net training on Google Colab

Trains the oil-slick segmentation model on the Trujillo-Acatitla Sentinel-1 dataset.

**Runtime → Change runtime type → T4 GPU** pehle set kar lena, warna CPU pe chalega aur bahut slow hoga.

---

### Ye notebook kya karta hai

1. GPU check karta hai
2. Tumhara `ml/` folder Google Drive se load karta hai
3. Zenodo se dataset download karta hai (Part III, ~9.9 GB) — **Colab ki local disk pe**, Drive pe nahi
4. U-Net train karta hai
5. Checkpoint + manifest **Drive pe** save karta hai, taaki session disconnect hone pe bhi na khoye

### Do cheezein jaan lo

- **Dataset local disk pe jaata hai, checkpoints Drive pe.** Dataset har session dobara download karna padega (~10 min), lekin Drive se hazaaron tiles read karna dataloader ko itna slow kar deta hai ki training hi bottleneck ban jaati hai. Checkpoints chhote hain, isliye wo Drive pe safe rehte hain.
- **Free Colab ~12 ghante baad session kaat deta hai** aur idle hone pe usse bhi jaldi. Training har epoch pe checkpoint likhti hai, toh disconnect hone pe latest `best.pt` Drive mein mil jaayega.


## 1 — GPU check


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU — Runtime > Change runtime type > T4 GPU')

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))


## 2 — Drive mount karo

Apne local machine se do cheezein Drive pe chahiye: `ml/` folder (training code) aur
`backend/src/spilltrace/` package (isme SAR preprocess + tiling hai jo trainer use karta hai —
sirf numpy chahiye ise, koi heavy dependency nahi).

Local terminal pe, repo root se:

```bash
cd /home/venom/Desktop/SIH-2026
zip -r sih_train.zip ml backend/src/spilltrace \
    -x 'ml/data/*' 'ml/runs/*' '*/__pycache__/*'
```

`sih_train.zip` ko Drive pe `MyDrive/SIH-2026/` mein upload karo. Neeche wali cell khud unzip kar degi.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib, zipfile
ROOT = pathlib.Path('/content/drive/MyDrive/SIH-2026')
ML_DIR = ROOT / 'ml'
PKG_DIR = ROOT / 'backend' / 'src' / 'spilltrace'

zip_path = ROOT / 'sih_train.zip'
if (not ML_DIR.exists() or not PKG_DIR.exists()) and zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(ROOT)
    print('unzipped')

assert ML_DIR.exists(), f'{ML_DIR} nahi mila — upload check karo'
assert PKG_DIR.exists(), f'{PKG_DIR} nahi mila — zip me backend/src/spilltrace include karna tha'
print('ml/ ok     :', sorted(p.name for p in ML_DIR.iterdir())[:8])
print('spilltrace :', sorted(p.name for p in PKG_DIR.iterdir())[:8])


## 3 — Dependencies

Colab mein `torch`, `numpy`, `scipy`, `PyYAML` pehle se hote hain. `rasterio` (GeoTIFF padhne ke liye)
aur `py7zr` (dataset 7-Zip archives) install karne padte hain.


In [ ]:
!pip install -q rasterio py7zr PyYAML
print('done')


## 4 — Working directory set karo

Code Drive se hi run hota hai (chhote Python files hain, I/O koi issue nahi), lekin **data aur runs local disk pe**.


In [ ]:
import os, sys, pathlib

os.chdir(ML_DIR)
sys.path.insert(0, str(ML_DIR / 'src'))                 # training code
sys.path.insert(0, str(ROOT / 'backend' / 'src'))        # spilltrace.ml.preprocess / .tiling

# sanity: the one cross-package import the trainer needs
from spilltrace.ml.preprocess import clip_and_standardise  # noqa: F401
from spilltrace.ml.tiling import tile_array                # noqa: F401
print('spilltrace.ml import OK')

DATA_DIR = pathlib.Path('/content/data/trujillo')                 # local disk — fast
RUNS_DIR = pathlib.Path('/content/drive/MyDrive/SIH-2026/runs')   # Drive — persistent
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('cwd  :', os.getcwd())
print('data :', DATA_DIR)
print('runs :', RUNS_DIR)
!df -h /content | tail -1


## 5 — Dataset

**Do raaste:**

**(a) Sabse simple** — yeh notebook Zenodo se **Part III** (9.86 GB: 150 oil + 150 look-alike +
150 no-oil, masks ke saath) khud download kar lega. Free Colab ke liye best.

**(b) Agar tumne Part I / II already download kiye hain** — unka ek chhota subset local pe
`prepare_local_dataset.py` se banao (guide PDF me steps hain), `images/` + `masks/` wala folder
ek zip me daal ke Drive pe upload karo `MyDrive/SIH-2026/trujillo_subset.zip`, phir yeh chalao:

```python
import zipfile, pathlib
z = pathlib.Path('/content/drive/MyDrive/SIH-2026/trujillo_subset.zip')
if z.exists():
    with zipfile.ZipFile(z) as f: f.extractall(DATA_DIR)
    print('subset ready:', sorted(p.name for p in DATA_DIR.iterdir()))
```

Aur config me `data.root` ko `/content/data/trujillo` pe point karo (colab_gpu.yaml me already wahi hai).

Neeche (a) wala raasta hai. (b) use kar rahe ho toh agli do cell skip kar do.


In [ ]:
!python scripts/download_dataset.py --parts III --dry-run


Ab actual download (~10-15 min):


In [ ]:
!python scripts/download_dataset.py --parts III --dest /content/data/trujillo

!du -sh /content/data/trujillo
!find /content/data/trujillo -name '*.tif' | head -5
!find /content/data/trujillo -name '*.tif' | wc -l


> **Disk bhar gaya?** Free Colab ~107 GB deta hai, Part III ke liye kaafi hai. Poora dataset (96.5 GB) free tier pe mat try karna. Aur kam chahiye toh: `--max-images 100`


## 6 — Ek epoch ka benchmark (pehle yeh karo)

Poori training shuru karne se pehle **ek epoch chala ke dekho kitna time lagta hai**. 40 epochs commit karne se pehle pata chal jaayega ki total kitna lagega.

Yeh `docs/DECISIONS.md` AD-11 ka rule hai: *"benchmark one epoch before committing"* — number maapo, andaaza mat lagao.


In [ ]:
!python src/train.py --config configs/colab_gpu.yaml --benchmark-epoch


Upar jo per-epoch time dikha, usse 40 se multiply karo. Agar 12 ghante se zyada aa raha hai toh `configs/colab_gpu.yaml` mein `epochs` kam kar do, ya `data.max_images` set kar do.


## 7 — Training

Checkpoints seedha Drive pe jaate hain, toh session kat bhi jaaye toh kaam nahi khoyega.


In [ ]:
!python src/train.py \
    --config configs/colab_gpu.yaml \
    --output-dir /content/drive/MyDrive/SIH-2026/runs


## 8 — Results dekho

Metrics **oil class ke liye** report hote hain — mean over classes nahi, kyunki us mein `sea` aur `land` ke trivially-easy pixels headline ko phula dete hain (AD-10).


In [ ]:
import json, pathlib

run_dir = pathlib.Path('/content/drive/MyDrive/SIH-2026/runs/unet-colab-gpu')
print('files:', sorted(p.name for p in run_dir.iterdir()))

manifest = json.loads((run_dir / 'manifest.json').read_text())
print()
print('=== MEASURED METRICS (oil class) ===')
print(json.dumps(manifest.get('metrics', {}), indent=2))
print()
print('seed          :', manifest.get('seed'))
print('split hash    :', manifest.get('split_hash'))
print('epochs run    :', manifest.get('epochs_completed'))
print('device        :', manifest.get('device'))


### Realistic expectations

`docs/ML_PIPELINE.md` §7 se — ye published ceilings hain, targets nahi:

| | oil-class Dice | oil-class IoU |
|---|---|---|
| Is config se honest expectation | 0.60 – 0.75 | 0.45 – 0.60 |
| Dataset paper (GPU, unka apna split) | — | 0.96 |
| Independent group, same data | 0.865 | 0.921 |

Jo bhi tum maapo, wahi report karna. Paper ke numbers unke split aur preprocessing pe hain — humare split se comparable nahi hain.


## 9 — Checkpoint local machine pe le jao

Do options:


In [ ]:
# Option A — Drive se seedha download (browser)
from google.colab import files
import shutil

shutil.make_archive('/content/unet-trained', 'zip', run_dir)
files.download('/content/unet-trained.zip')


In [ ]:
# Option B — Drive pe hi rehne do, local machine pe Drive sync/download kar lo
!ls -lh /content/drive/MyDrive/SIH-2026/runs/unet-colab-gpu/


---

## 10 — Aage kya (yeh local machine pe karna hai)

`register_model.py` ko database aur MinIO chahiye, jo Colab se reachable nahi hain. Toh checkpoint local pe laane ke baad:

```bash
cd /home/venom/Desktop/SIH-2026

# 1. Stack chalu ho
make up

# 2. Checkpoint register karo
python ml/scripts/register_model.py \
    --checkpoint ml/runs/unet-colab-gpu/best.pt \
    --manifest   ml/runs/unet-colab-gpu/manifest.json \
    --version 0.1.0 \
    --activate

# 3. Backend ko batao ki ab U-Net use kare
#    .env mein:  SPILLTRACE_SEGMENTATION_MODEL=unet
#    aur torch wali image chahiye:
make build-ml
make up
```

**Note:** `--activate` tabhi kaam karega jab manifest mein actual measured metrics hon. Agar metrics khaali hain toh script activate karne se mana kar degi — ye jaanbujh ke hai (AD-10): ek unevaluated model register toh ho sakta hai, lekin chupke se production model nahi ban sakta.
